# GPS Module

This notebook shows how to use the Adafruit Ultimate GPS Breakout v3 module connected to the board via UART. The GPS data is read using the Python serial library (pyserial) and displayed in real time. Each sentence follows the standard NMEA format, which can be parsed to extract information such as latitude, longitude, and GPS fix status.

The GPS module used is the Adafruit Ultimate GPS Breakout v3, which communicates at a default baud rate of 9600 bps and outputs data through its TX pin.

#### References
https://www.adafruit.com/product/746
https://learn.adafruit.com/adafruit-ultimate-gps
https://pyserial.readthedocs.io/en/latest/

In [1]:
gps_enabled = True  # Set to False to stop reading GPS data

from pynq import Overlay, MMIO
from time import sleep

In [2]:
ol = Overlay("car.bit", ignore_version=True)
print("✔ Overlay 'car.bit' loaded successfully.\n")

✔ Overlay 'car.bit' loaded successfully.



In [3]:
# Access the AXI GPIO block (used for GPS enable/reset)
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000, debug=False)

In [4]:
# Bit definitions for your GPS control pins:
# bit0 = GPS Enable
# bit1 = GPS Reset
GPIO_EN = 1 << 0
GPIO_RST = 1 << 1

# Enable GPS power and release reset
gpio.write(0x00, GPIO_EN)
sleep(0.1)
gpio.write(0x00, GPIO_EN | GPIO_RST)

print("✔ GPS module powered and initialized via GPIO.")


✔ GPS module powered and initialized via GPIO.


In [5]:
# ----------------------------- GPS Coordinate Parsing -----------------------------
def nmea_to_decimal(coord, direction):
    """Converts NMEA DDMM.MMMM to decimal degrees."""
    if not coord or not direction:
        return None
    degrees = int(coord[:2])
    minutes = float(coord[2:])
    decimal = degrees + minutes/60
    if direction in ['S', 'W']:
        decimal *= -1
    return round(decimal, 6)

def parse_gps_line(line):
    """Parses and prints coordinates from NMEA sentence."""
    if line.startswith("$GPGGA") or line.startswith("$GPRMC"):
        parts = line.split(',')
        if len(parts) > 5 and parts[2] and parts[4]:
            lat = nmea_to_decimal(parts[2], parts[3])
            lon = nmea_to_decimal(parts[4], parts[5])
            print(f"Latitude: {lat}, Longitude: {lon}")

# Example test with known data
parse_gps_line("$GPGGA,214500.00,3402.1234,N,11814.5678,W,1,07,1.0,123.4,M,-34.0,M,,*65")


Latitude: 34.03539, Longitude: -24.57613


In [6]:
#raw data dont run
from time import sleep
from IPython.display import clear_output

gps_uart = ol.gps_uart

def read_nmea_sentence():
    text = ""
    for _ in range(100):  # read up to 100 bytes at a time
        status = gps_uart.read(0x08)
        if status & 0x01:             # data valid
            val = gps_uart.read(0x00)
            text += chr(val & 0xFF)
            sleep(0.002)              # small delay to not overflow
        else:
            break
    return text

try:
    while True:
        sentence = read_nmea_sentence()
        if sentence:
            clear_output(wait=True)
            print("🌍 GPS Receiving Data:\n")
            print(sentence)
        sleep(0.5)

except KeyboardInterrupt:
    print("\n🛑 GPS reading stopped by user.")


🌍 GPS Receiving Data:

$GPGGA,000039.09

🛑 GPS reading stopped by user.


In [11]:
#Original With Parsed Data Dont run
from time import sleep
from IPython.display import clear_output

gps_uart = ol.gps_uart

# ----------------------------- Read NMEA Sentences -----------------------------
def read_nmea_sentence():
    text = ""
    for _ in range(200):  # read more bytes to capture full sentences
        status = gps_uart.read(0x08)
        if status & 0x01:  # data valid
            val = gps_uart.read(0x00)
            text += chr(val & 0xFF)
            sleep(0.002)
        else:
            break
    return text

# ----------------------------- NMEA Parser Helpers -----------------------------
def nmea_to_decimal(coord, direction):
    """Convert NMEA DDMM.MMMM to decimal degrees."""
    if not coord or not direction:
        return None
    try:
        degrees = int(coord[:2])
        minutes = float(coord[2:])
        decimal = degrees + minutes / 60.0
        if direction in ['S', 'W']:
            decimal *= -1
        return round(decimal, 6)
    except:
        return None

def parse_gpgga(sentence):
    """Parse GPGGA for time, coordinates, satellites, and altitude."""
    if sentence.startswith("$GPGGA"):
        parts = sentence.split(',')
        if len(parts) >= 10:
            utc_time = parts[1]
            lat = nmea_to_decimal(parts[2], parts[3])
            lon = nmea_to_decimal(parts[4], parts[5])
            fix_quality = parts[6]
            sats = parts[7]
            alt = parts[9]

            print(f"🕒 UTC Time: {utc_time}")
            print(f"📍 Latitude: {lat}")
            print(f"📍 Longitude: {lon}")
            print(f"📡 Fix Quality: {fix_quality} (0 = none, 1 = GPS fix)")
            print(f"🛰️ Satellites: {sats}")
            print(f"🌄 Altitude: {alt} m")

# ----------------------------- Continuous Read + Parse -----------------------------
try:
    while True:
        sentence = read_nmea_sentence()
        if sentence and "$GPGGA" in sentence:
            clear_output(wait=True)
            print("🌍 GPS Receiving Data:\n")
            print(sentence)
            print("\nParsed Data:")
            for line in sentence.splitlines():
                if line.startswith("$GPGGA"):
                    parse_gpgga(line)
        sleep(0.5)

except KeyboardInterrupt:
    print("\n🛑 GPS reading stopped by user.")


🌍 GPS Receiving Data:

$GPGGA,000232.09

Parsed Data:

🛑 GPS reading stopped by user.


In [7]:
#New, Run
from time import sleep
from IPython.display import clear_output

gps_uart = ol.gps_uart

# --- Convert NMEA DDMM.MMMM to decimal degrees ---
def nmea_to_decimal(coord, direction):
    if not coord or not direction:
        return None
    try:
        degrees = int(coord[:2])
        minutes = float(coord[2:])
        decimal = degrees + minutes / 60.0
        if direction in ['S', 'W']:
            decimal *= -1
        return round(decimal, 6)
    except:
        return None

# --- Read raw NMEA stream from GPS UART ---
def read_nmea_sentence():
    text = ""
    for _ in range(120):
        status = gps_uart.read(0x08)       # UART status
        if status & 0x01:                  # data available flag
            val = gps_uart.read(0x00)      # read one byte
            text += chr(val & 0xFF)
            sleep(0.002)
        else:
            break
    return text

# --- Parse $GPGGA line for lat/lon/fix/sats ---
def parse_gpgga(sentence):
    parts = sentence.split(',')
    if len(parts) < 8:
        return None, None, None, None

    lat = nmea_to_decimal(parts[2], parts[3])
    lon = nmea_to_decimal(parts[4], parts[5])
    fix = parts[6]      # 0 = no fix, 1 = GPS fix, 2 = DGPS
    sats = parts[7]     # number of satellites in use

    return lat, lon, fix, sats

# --- Continuous monitor loop ---
try:
    last_lat = None
    last_lon = None
    last_fix = None
    last_sats = None

    print("📡 Listening for GPS... (go outside for lock)\n")

    while True:
        sentence = read_nmea_sentence()

        if sentence:
            clear_output(wait=True)
            print("🌍 RAW GPS Data:\n")
            print(sentence)

            if "$GPGGA" in sentence:
                for line in sentence.splitlines():
                    if line.startswith("$GPGGA"):
                        lat, lon, fix, sats = parse_gpgga(line)

                        if lat is not None:
                            last_lat = lat
                        if lon is not None:
                            last_lon = lon
                        if fix is not None:
                            last_fix = fix
                        if sats is not None:
                            last_sats = sats

            print("\n📌 Parsed GPS Info:")
            print(f"Latitude:     {last_lat}")
            print(f"Longitude:    {last_lon}")
            print(f"Fix Quality:  {last_fix}  (0=no lock, 1=GPS fix)")
      xt33333333333c ♠3      print(f"Satellites:   {last_sats}")

        sleep(0.5)

except KeyboardInterrupt:
    print("\n🛑 GPS reading stopped by user.")


IndentationError: unindent does not match any outer indentation level (<ipython-input-7-47d9473d6677>, line 82)

In [8]:
from time import sleep
from IPython.display import clear_output

gps_uart = ol.gps_uart

# --- Convert NMEA DDMM.MMMM to decimal degrees ---
def nmea_to_decimal(coord, direction):
    if not coord or not direction:
        return None
    try:
        degrees = int(coord[:2])
        minutes = float(coord[2:])
        decimal = degrees + minutes / 60.0
        if direction in ['S', 'W']:
            decimal *= -1
        return round(decimal, 6)
    except:
        return None

# --- Read raw NMEA stream from GPS UART ---
def read_nmea_sentence():
    text = ""
    for _ in range(120):
        status = gps_uart.read(0x08)       # UART status
        if status & 0x01:                  # data available
            val = gps_uart.read(0x00)      # read one byte
            text += chr(val & 0xFF)
            sleep(0.002)
        else:
            break
    return text

# --- Parse $GPGGA line for lat/lon/fix/sats ---
def parse_gpgga(sentence):
    parts = sentence.split(',')
    if len(parts) < 8:
        return None, None, None, None

    lat = nmea_to_decimal(parts[2], parts[3])
    lon = nmea_to_decimal(parts[4], parts[5])
    fix = parts[6]      # 0 = no fix, 1 = GPS fix, 2 = DGPS
    sats = parts[7]     # satellites in use

    return lat, lon, fix, sats

# --- Continuous monitor loop ---
try:
    last_lat = None
    last_lon = None
    last_fix = None
    last_sats = None

    print("📡 Listening for GPS... (go outside for lock)\n")

    while True:
        sentence = read_nmea_sentence()

        if sentence:
            clear_output(wait=True)
            print("🌍 RAW GPS Data:\n")
            print(sentence)

            # Parse GPGGA if available
            if "$GPGGA" in sentence:
                for line in sentence.splitlines():
                    if line.startswith("$GPGGA"):
                        lat, lon, fix, sats = parse_gpgga(line)

                        if lat is not None:
                            last_lat = lat
                        if lon is not None:
                            last_lon = lonX
                        if fix is not None:
                            last_fix = fix
                        if sats is not None:
                            last_sats = sats

            print("\n📌 Parsed GPS Info:")
            print(f"Latitude:     {last_lat}")
            print(f"Longitude:    {last_lon}")
            print(f"Fix Quality:  {last_fix}  (0=no lock, 1=GPS fix)")
            print(f"Satellites:   {last_sats}")

        sleep(0.5)

except KeyboardInterrupt:
    print("\n🛑 GPS reading stopped by user.")


🌍 RAW GPS Data:

$GPGGA,223025.007,,13.33W,715,6.,,3.,,*8
PS,,,02,71,70,3,,17,.009*8
GRC23200A31.54N18174,,0,.0212,,*1
GVG00T,,.3N00,,*8



NameError: name 'lonX' is not defined

In [15]:
from time import sleep
from IPython.display import clear_output

gps_uart = ol.gps_uart

# --- Convert NMEA DDMM.MMMM to decimal degrees ---
def nmea_to_decimal(coord, direction):
    if not coord or not direction:
        return None
    try:
        degrees = int(coord[:2])
        minutes = float(coord[2:])
        decimal = degrees + minutes / 60.0
        if direction in ['S', 'W']:
            decimal *= -1
        return round(decimal, 6)
    except:
        return None

# --- Read raw NMEA stream from GPS UART ---
def read_nmea_sentence():
    text = ""
    for _ in range(120):
        status = gps_uart.read(0x08)
        if status & 0x01:  
            val = gps_uart.read(0x00)
            text += chr(val & 0xFF)
            sleep(0.002)
        else:
            break
    return text

# --- Parse $GPGGA line for lat/lon/fix/sats ---
def parse_gpgga(sentence):
    parts = sentence.split(',')
    if len(parts) < 8:
        return None, None, None, None

    lat = nmea_to_decimal(parts[2], parts[3])
    lon = nmea_to_decimal(parts[4], parts[5])
    fix = parts[6]
    sats = parts[7]

    return lat, lon, fix, sats

# --- Continuous monitor loop ---
try:
    last_lat = None
    last_lon = None
    last_fix = None
    last_sats = None

    print("📡 Listening for GPS... (go outside for lock)\n")

    while True:
        sentence = read_nmea_sentence()

        if sentence:
            clear_output(wait=True)
            print("🌍 RAW GPS Data:\n")
            print(sentence)

            if "$GPGGA" in sentence:
                for line in sentence.splitlines():
                    if line.startswith("$GPGGA"):
                        lat, lon, fix, sats = parse_gpgga(line)

                        if lat is not None:
                            last_lat = lat
                        if lon is not None:
                            last_lon = lon     # <-- FIXED HERE
                        if fix is not None:
                            last_fix = fix
                        if sats is not None:
                            last_sats = sats

            print("\n📌 Parsed GPS Info:")
            print(f"Latitude:     {last_lat}")
            print(f"Longitude:    {last_lon}")
            print(f"Fix Quality:  {last_fix}")
            print(f"Satellites:   {last_sats}")

        sleep(0.5)

except KeyboardInterrupt:
    print("\n🛑 GPS reading stopped by user.")


🌍 RAW GPS Data:

$GPGGA,224137.00

📌 Parsed GPS Info:
Latitude:     34.0
Longitude:    91.0
Fix Quality:  615
Satellites:   D4

🛑 GPS reading stopped by user.


In [8]:
# FINAL WORKING GPS CODE – Run this now others 
from time import sleep
from IPython.display import clear_output, HTML

gps_uart = ol.gps_uart

# FIXED FUNCTION – handles both latitude (2 digits) and longitude (3 digits) perfectly
def nmea_to_decimal(coord, direction):
    if not coord or not direction:
        return None
    try:
        if '.' not in coord:
            coord += '.0'  # safety
        
        whole, decimal_part = coord.split('.')
        whole = whole.lstrip('0') or '0'  # remove leading zeros but keep 0
        
        if len(whole) >= 3 and direction in ['E', 'W']:  # longitude case
            degrees = int(whole[:-2])           # last 2 digits of whole = minutes part
            minutes = float(whole[-2:] + '.' + decimal_part)
        else:                                           # latitude case
            degrees = int(whole[:-2] or '0')
            minutes = float(whole[-2:] + '.' + decimal_part) if len(whole) >= 2 else float('0.' + decimal_part)
        
        decimal = degrees + minutes / 60.0
        if direction in ['S', 'W']:
            decimal = -decimal
        return round(decimal, 6)
    except:
        return None

# Read whatever is waiting in the UART buffer
def read_nmea_sentence():
    text = ""
    for _ in range(200):
        status = gps_uart.read(0x08)
        if status & 0x01:
            val = gps_uart.read(0x00)
            text += chr(val & 0xFF)
            sleep(0.001)
        else:
            break
    return text

# Parse only clean GPGGA sentences
def parse_gpgga(sentence):
    if not sentence.startswith('$GPGGA'):
        return None
    parts = sentence.split(',')
    if len(parts) < 15:
        return None
    lat = nmea_to_decimal(parts[2], parts[3])
    lon = nmea_to_decimal(parts[4], parts[5])
    fix = parts[6]
    sats = parts[7].zfill(2)
    alt = parts[9] if parts[9] else ""
    return lat, lon, fix, sats, alt

# Main loop
try:
    last_lat = None
    last_lon = None
    last_fix = None
    last_sats = None
    last_alt = None

    print("Waiting for GPS lock in Los Angeles...\n")
    while True:
        raw = read_nmea_sentence()
        if raw:
            clear_output(wait=True)
            print("RAW GPS Data:\n")
            print(raw.strip())

            lines = [l for l in raw.splitlines() if l.startswith('$GPGGA')]
            if lines:
                result = parse_gpgga(lines[-1])
                if result:
                    lat, lon, fix, sats, alt = result
                    if lat is not None: last_lat = lat
                    if lon is not None: last_lon = lon
                    if fix: last_fix = fix
                    if sats: last_sats = sats
                    if alt: last_alt = alt

            print("\nParsed GPS Info:")
            print(f"Latitude:     {last_lat if last_lat is not None else '-'}")
            print(f"Longitude:    {last_lon if last_lon is not None else '-'}")
            print(f"Fix Quality:  {last_fix if last_fix else '-'}  (0=no lock, 1=GPS fix)")
            print(f"Satellites:   {last_sats if last_sats else '-'}")
            print(f"Altitude:     {last_alt + ' m' if last_alt else '-'}")

            if last_lat and last_lon and last_lon != 0:
                url = f"https://www.google.com/maps?q={last_lat},{last_lon}"
                display(HTML(f'<h3><a href="{url}" target="_blank">Open in Google Maps</a></h3>'))

        sleep(0.5)

except KeyboardInterrupt:
    print("\nGPS reading stopped by user.")

RAW GPS Data:

$GPGGA,235223.002,T,,M,.40,N,0.74K,D*0F

Parsed GPS Info:
Latitude:     34.240885
Longitude:    -118.529012
Fix Quality:  2  (0=no lock, 1=GPS fix)
Satellites:   10
Altitude:     272.7 m



GPS reading stopped by user.


In [9]:
# FINAL CODE – Los Angeles GPS with dual Street View + 3D Earth links
from time import sleep
from IPython.display import clear_output, HTML

gps_uart = ol.gps_uart

# Perfectly working NMEA → decimal degrees (handles 118° West correctly)
def nmea_to_decimal(coord, direction):
    if not coord or not direction:
        return None
    try:
        if '.' not in coord:
            coord += '.0'
        whole, frac = coord.split('.')
        whole = whole.lstrip('0') or '0'

        if len(whole) >= 3 and direction in 'EW':  # longitude
            degrees = int(whole[:-2])
            minutes = float(whole[-2:] + '.' + frac)
        else:  # latitude
            if len(whole) < 2:
                whole = '0' + whole
            degrees = int(whole[:-2] or '0')
            minutes = float((whole[-2:] if len(whole) >= 2 else '00') + '.' + frac)

        decimal = degrees + minutes / 60.0
        if direction in 'SW':
            decimal = -decimal
        return round(decimal, 6)
    except:
        return None

def read_nmea_sentence():
    text = ""
    for _ in range(200):
        status = gps_uart.read(0x08)
        if status & 0x01:
            val = gps_uart.read(0x00)
            text += chr(val & 0xFF)
            sleep(0.001)
        else:
            break
    return text

def parse_gpgga(sentence):
    if not sentence.startswith('$GPGGA'):
        return None
    parts = sentence.split(',')
    if len(parts) < 15:
        return None
    lat = nmea_to_decimal(parts[2], parts[3])
    lon = nmea_to_decimal(parts[4], parts[5])
    fix = parts[6]
    sats = parts[7].zfill(2)
    alt = parts[9] if parts[9] else ""
    return lat, lon, fix, sats, alt

# Main loop
try:
    last_lat = last_lon = last_fix = last_sats = last_alt = None

    print("Waiting for GPS lock in LA... (both Street View & 3D ready!)\n")
    while True:
        raw = read_nmea_sentence()
        if raw:
            clear_output(wait=True)
            print("RAW GPS Data:\n" + raw.strip())

            for line in raw.splitlines():
                if line.startswith('$GPGGA'):
                    result = parse_gpgga(line)
                    if result:
                        lat, lon, fix, sats, alt = result
                        if lat:  last_lat = lat
                        if lon:  last_lon = lon
                        if fix:  last_fix = fix
                        if sats: last_sats = sats
                        if alt:  last_alt = alt

            print("\nParsed GPS Info:")
            print(f"Latitude:     {last_lat or '-'}")
            print(f"Longitude:    {last_lon or '-'}")
            print(f"Fix Quality:  {last_fix or '-'}  (0=no lock, 1=GPS fix)")
            print(f"Satellites:   {last_sats or '-'}")
            print(f"Altitude:     {(last_alt + ' m') if last_alt else '-'}")

            # DUAL MAGIC LINKS
            if last_lat and last_lon and last_lon != 0:
                street_view = f"https://www.google.com/maps/@{last_lat},{last_lon},3a,75y,90t/data=!3m6!1e1!3m4!1s0!2e0!7i16384!8i8192"
                earth_3d    = f"https://www.google.com/maps/@{last_lat},{last_lon},3000a,35y,90h,1t/data=!3m1!1e3"

                display(HTML(f'''
                <h3>
                    <a href="{street_view}" target="_blank">Street View (stand here!)</a> 
                    &nbsp;&nbsp;|&nbsp; 
                    <a href="{earth_3d}" target="_blank">3D Earth Fly-over</a>
                </h3>
                '''))

        sleep(0.5)

except KeyboardInterrupt:
    print("\nGPS stopped!")

RAW GPS Data:
$GPGGA,000006.00

Parsed GPS Info:
Latitude:     0.05
Longitude:    0.016667
Fix Quality:  059  (0=no lock, 1=GPS fix)
Satellites:   25
Altitude:     17 m



GPS stopped!
